In [2]:
import pandas as pd
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

df = pd.read_csv("../data/olist_orders_clean.csv")
print("Loaded shape:", df.shape)
df.head()

Loaded shape: (96470, 22)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,is_late,...,review_score,review_answer_timestamp,n_items,total_price,total_freight,n_distinct_sellers,total_payment_value,payment_methods_used,max_installments,is_negative_review
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-7.107488,False,...,4.0,2017-10-12 03:43:48,1,29.99,8.72,1,38.71,2.0,1.0,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-5.355729,False,...,4.0,2018-08-08 18:37:50,1,118.70,22.76,1,141.46,1.0,1.0,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-17.245498,False,...,5.0,2018-08-22 19:07:58,1,159.90,19.22,1,179.12,1.0,3.0,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,-12.980069,False,...,5.0,2017-12-05 19:21:58,1,45.00,27.20,1,72.20,1.0,1.0,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,-9.238171,False,...,5.0,2018-02-18 13:02:51,1,19.90,8.72,1,28.62,1.0,1.0,False


In [3]:
has_review = df["review_score"].notna()
late_with_review = df[has_review & df["is_late"]]

baseline_rate = late_with_review["is_negative_review"].mean()
n_late_orders = len(late_with_review)

print(f"Baseline negative review rate among late orders: {baseline_rate:.4f} ({baseline_rate*100:.1f}%)")
print(f"Number of late orders with a review (our eligible population size so far): {n_late_orders}")

Baseline negative review rate among late orders: 0.5407 (54.1%)
Number of late orders with a review (our eligible population size so far): 7661


In [4]:
power_calc = NormalIndPower()

mdes = [0.03, 0.05, 0.08, 0.10, 0.15]

print(f"{'MDE (pts)':>10} {'treated rate':>13} {'n per arm':>12} {'total n':>10}")
for mde in mdes:
    treatment_rate = baseline_rate - mde
    effect_size = proportion_effectsize(baseline_rate, treatment_rate)
    n_per_arm = power_calc.solve_power(effect_size=effect_size, alpha=0.05, power=0.8, alternative='two-sided')
    print(f"{mde:>10.2f} {treatment_rate:>13.3f} {n_per_arm:>12.0f} {n_per_arm*2:>10.0f}")

 MDE (pts)  treated rate    n per arm    total n
      0.03         0.511         4348       8695
      0.05         0.491         1567       3134
      0.08         0.461          612       1224
      0.10         0.441          391        782
      0.15         0.391          172        345


In [5]:
# Estimate the dataset's time span to get a monthly late-order rate
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])
date_range_days = (df["order_purchase_timestamp"].max() - df["order_purchase_timestamp"].min()).days
months_span = date_range_days / 30.44  # average days per month

late_orders_per_month = n_late_orders / months_span
print(f"Dataset spans approximately {months_span:.1f} months")
print(f"Late orders per month: {late_orders_per_month:.0f}")
print()

print(f"{'MDE (pts)':>10} {'total n needed':>16} {'est. months to run':>20}")
for mde in mdes:
    treatment_rate = baseline_rate - mde
    effect_size = proportion_effectsize(baseline_rate, treatment_rate)
    n_per_arm = power_calc.solve_power(effect_size=effect_size, alpha=0.05, power=0.8, alternative='two-sided')
    total_n = n_per_arm * 2
    est_months = total_n / late_orders_per_month
    print(f"{mde:>10.2f} {total_n:>16.0f} {est_months:>20.1f}")

Dataset spans approximately 23.4 months
Late orders per month: 327

 MDE (pts)   total n needed   est. months to run
      0.03             8695                 26.6
      0.05             3134                  9.6
      0.08             1224                  3.7
      0.10              782                  2.4
      0.15              345                  1.1


## Summary

This notebook derives the power analysis and sample size requirements for the proposed
late-delivery notification experiment.

**Key findings:**
- Baseline negative review rate among late orders: 54.1 percent (n=7,661 late orders with a review)
- Dataset spans approx. 23.4 months, giving approx. 327 late orders/month as the real-world
  experiment eligible-population rate
- Tested MDEs from 3-15 percentage points:
  - Small effects (3-5pt) require 3,100-8,700 orders total, implying 9.6-26.6 months of
    runtime -- not practically actionable
  - 8pt MDE requires approx. 1,224 total orders, approx. 3.7 months -- the chosen target:
    detectable in a reasonable quarter-length window, and large enough to matter to the business

**Decision:** target an 8 percentage point MDE, requiring approx. 612 customers per arm
(randomized by `customer_unique_id`, per the design doc), with an estimated 4 month runtime
given current late-order volume.

**Next notebook:** simulate the experiment with a known, heterogeneous treatment effect, and
validate that a standard statistical test correctly recovers it.